# Revision Fix 5 + Fix 11 on Kaggle T4 x2

Run this notebook in a fresh Kaggle session with **Internet ON** and **GPU: T4 x2**. It runs only Fix 5 and Fix 11. Fix 5 uses GPU0/Ollama port 11434; Fix 11 uses GPU1/Ollama port 11435. The wrapper prints heartbeat lines even when installs, downloads, or generation are quiet.

In [ ]:
import os
import pathlib
import subprocess
import time

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'rag-hallucination-detection'
REMOTE = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'

def run(cmd, cwd=None, check=True):
    print('\n>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)

print('START', time.ctime(), flush=True)
run(['nvidia-smi'], check=False)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--progress', '--branch', 'main', REMOTE, str(REPO)], cwd=WORK)
else:
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=False)
    run(['git', '-C', str(REPO), 'checkout', 'main'], check=False)
    run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=False)

os.chdir(REPO)
run(['git', 'log', '-1', '--oneline'])
run(['ls', '-lh', 'scripts/kaggle_fix5_11_t4x2.sh', 'scripts/kaggle_stream_fix5_11_t4x2.py'])

## 1. Setup + Ollama Preflight

This installs dependencies, starts two isolated Ollama servers, pulls `mistral`, checks both ports, and compiles the Fix 5 / Fix 11 scripts. Expected time: 15-35 minutes the first time because Mistral downloads.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage setup --heartbeat 15

## 2. Run Fix 5 + Fix 11 in Parallel

This is the main run. Expected time on T4 x2: roughly 2.5-4.5 hours. It prints wrapper heartbeats plus script heartbeats with CSV row counts.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage parallel --heartbeat 30

## 3. Status Check

Run this after the main run, or while debugging after an interruption. It only counts files and prints paths.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage status --heartbeat 15

## 4. Package Outputs

Download `/kaggle/working/fix5_11_t4x2_outputs.zip` from the right sidebar after this cell finishes.

In [ ]:
!python -u scripts/kaggle_stream_fix5_11_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix5_11_t4x2_outputs.zip

## Emergency Commands

If the main run is interrupted, do **not** delete the session immediately. Run the status cell first. If Ollama looks bad, rerun Setup, then rerun the parallel cell.